In [ ]:
import datetime as dt

import geopandas as gpd
import pandas as pd
from meteora import settings
from meteora import utils as meteora_utils
from meteora.clients import MeteoSwissClient

In [ ]:
# spatial extent: LV95 bounds (west, south, east, north) of the LCS study extent,
# rounded outward to the km. ACHTUNG: the study extent itself is derived from the LCS
# stations, which depend on the hot days detected here, so it cannot be an input.
# These bounds select the MeteoSwiss stations REH, SMA and UEB (the latter returns no
# temperature data and is dropped below).
region = [2674000, 1241000, 2691000, 1256000]

# NABEL data files (in data/raw)
nabel_stations_filepath = "../data/raw/nabel-stations.csv"
nabel_temp_filepath = "../data/raw/nabel-temp.csv"

# output files
dst_ts_df_filepath = "../data/interim/aws-ts-df.csv"
dst_stations_gdf_filepath = "../data/interim/aws-stations.gpkg"

# time range: JJA for each year
start_year = 2022
end_year = 2025
start_month = 6
end_month = 8

# hot-day selection parameters (select all days where max(station mean T) > threshold)
heatwave_t_threshold = 25
heatwave_n_consecutive_days = 1
station_agg_func = "mean"
inter_station_agg_func = "max"

# other
stations_crs = "epsg:2056"

In [ ]:
meteoswiss_client = MeteoSwissClient(region=region)
meteoswiss_ts_df = pd.concat(
    [
        meteoswiss_client.get_ts_df(
            settings.ECV_TEMPERATURE,
            dt.date(year, start_month, 1),
            dt.datetime.combine(
                dt.date(year, end_month + 1, 1) - dt.timedelta(days=1),
                dt.time.max,
            ),
        )
        for year in range(start_year, end_year + 1)
    ],
    axis="rows",
)
# convert to wide format (station x time)
meteoswiss_ts_df = meteora_utils.long_to_wide(meteoswiss_ts_df)
# strip timezone and resample to hourly means
if meteoswiss_ts_df.index.tz is not None:
    meteoswiss_ts_df.index = meteoswiss_ts_df.index.tz_convert(None)
meteoswiss_ts_df = meteoswiss_ts_df.resample("h").mean()
meteoswiss_ts_df.head()

In [ ]:
# NABEL CSV has 5 header lines (pollutant, aggregation, timezone, unit, source)
# then a header row "Date/time;StationA;StationB"
nabel_ts_df = pd.read_csv(
    nabel_temp_filepath,
    sep=";",
    encoding="latin1",
    skiprows=5,
    index_col="Date/time",
    parse_dates=True,
    dayfirst=True,
)
nabel_ts_df.index.name = "time"
# ACHTUNG: NABEL exports hourly means labelled at the END of the averaging interval
# and in MEZ/CET (UTC+1), whereas the MeteoSwiss series are start-labelled UTC. Both
# conventions compound, so the NABEL timestamps run two hours late: a 10-min lag scan
# against instantaneous METAR observations peaks at +100 min (CET-only would peak at
# +60). Shift them back so that both sources share the same clock.
nabel_ts_df.index = nabel_ts_df.index - pd.Timedelta(hours=2)
nabel_ts_df.head()

In [ ]:
# combine MeteoSwiss and NABEL into a single wide data frame
aws_ts_df = pd.concat([meteoswiss_ts_df, nabel_ts_df], axis="columns")
# drop stations that the client lists but for which no data was returned: an
# all-empty column would otherwise propagate into the station table and be counted
# as an additional AWS carrying no information
aws_ts_df = aws_ts_df.dropna(axis="columns", how="all")
# aws_ts_df.index.name = "time"

# select all hot days: detect per year separately to avoid cross-year period jumps
aws_ts_df = pd.concat(
    [
        meteora_utils.get_heatwave_ts_df(
            aws_ts_df[
                (aws_ts_df.index.year == year)
                & aws_ts_df.index.month.isin(range(start_month, end_month + 1))
            ],
            heatwave_t_threshold=heatwave_t_threshold,
            heatwave_n_consecutive_days=heatwave_n_consecutive_days,
            station_agg_func=station_agg_func,
            inter_station_agg_func=inter_station_agg_func,
        )
        for year in range(start_year, end_year + 1)
    ]
)
aws_ts_df.columns.name = "station_id"
aws_ts_df.head()

In [ ]:
# MeteoSwiss stations (index = station_id = station abbreviations)
meteoswiss_stations_gdf = meteoswiss_client.stations_gdf[["geometry"]].to_crs(
    stations_crs
)

# NABEL stations from local CSV (LV95 coordinates)
nabel_stations_df = pd.read_csv(nabel_stations_filepath, index_col="station_id")
nabel_stations_gdf = gpd.GeoDataFrame(
    nabel_stations_df[[]],
    geometry=gpd.points_from_xy(
        nabel_stations_df["lv_95_e"], nabel_stations_df["lv_95_n"]
    ),
    crs=stations_crs,
)

# combine and keep only stations present in the time series
aws_gdf = pd.concat([meteoswiss_stations_gdf, nabel_stations_gdf])
aws_gdf = aws_gdf.loc[aws_ts_df.columns.intersection(aws_gdf.index)]
aws_gdf.head()

In [ ]:
aws_ts_df.to_csv(dst_ts_df_filepath)
aws_gdf.reset_index().to_file(dst_stations_gdf_filepath)